# Genixcure — PCR_6_13 단일 샘플 Merged Unique Count

**대상:** `PCR_6_13_1.fq.gz` + `PCR_6_13_2.fq.gz` (Google Drive 원본)

**목표:** `fastp`의 기존 병합 옵션을 유지하면서 병합에 성공한 read의 **정확히 같은 서열**을 1개로 묶고, `sequence,read_count,length_bp`를 기록합니다. 이전 10개와 같은 원리로 처리합니다. Primer/index trimming은 하지 않습니다.

**저장:** `MyDrive/Genixcure_PCR_Merge_Analysis/All_10_Samples/PCR_6_13_merged_unique_sequences.csv.gz`

기존 PCR 10개 결과는 수정하지 않습니다. 원본 파일을 Drive에서 Colab 로컬 `/content`로 내려받아 재사용하고, 병합/정렬 후 임시 파일을 삭제합니다. 재실행 시 완전한 결과를 자동 검증하여 건너뜁니다.


In [ ]:
!command -v fastp >/dev/null || (apt-get update -qq >/dev/null && apt-get install -y -qq fastp >/dev/null)
!fastp --version


In [ ]:
from google.colab import auth,drive
auth.authenticate_user()
drive.mount('/content/drive')
import google.auth, os, re, json, hashlib, subprocess, shutil, time, gzip, csv
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

SAMPLE='PCR_6_13'
FOLDER_ID='12YerJ_CIPzXwpjOf6krNy8QKwH_zg1nt'
OUT=Path('/content/drive/MyDrive/Genixcure_PCR_Merge_Analysis/All_10_Samples')
TMP=Path('/content/pcr_merge_work/PCR_6_13')
OUT.mkdir(parents=True,exist_ok=True)
TMP.mkdir(parents=True,exist_ok=True)
RESULT=OUT/f'{SAMPLE}_merged_unique_sequences.csv.gz'
SUMMARY=OUT/f'{SAMPLE}_merge_summary.csv'
FASTP_THREADS=4
SORT_RAM='2G'
OVERWRITE=False
print('Result:',RESULT)


## FASTQ 목록 및 인증 다운로드

R1/R2 각각 약 533MB·601MB입니다. 파일 크기와 Drive MD5가 제공되면 다운로드 결과를 검증합니다. 동일한 완전 다운로드 파일은 재사용합니다.


In [ ]:
credentials,_=google.auth.default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
svc=build('drive','v3',credentials=credentials,cache_discovery=False)
resp=svc.files().list(
    q=f"'{FOLDER_ID}' in parents and trashed=false",
    fields='nextPageToken,files(id,name,size,md5Checksum)',
    pageSize=1000,supportsAllDrives=True,includeItemsFromAllDrives=True
).execute()
files=resp['files']
r1=[x for x in files if re.search(r'_1\.(?:fq|fastq)\.gz$',x['name'],re.I)]
r2=[x for x in files if re.search(r'_2\.(?:fq|fastq)\.gz$',x['name'],re.I)]
assert len(r1)==1 and len(r2)==1, (r1,r2)
print('R1:',r1[0]['name'],r1[0].get('size'))
print('R2:',r2[0]['name'],r2[0].get('size'))

def file_md5(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):
            h.update(block)
    return h.hexdigest()

def download(x):
    dest=TMP/x['name']
    expected=int(x['size'])
    if dest.exists() and dest.stat().st_size!=expected:
        dest.unlink()
    if not dest.exists():
        # Separate API client per thread (httplib2 is not thread-safe)
        creds,_=google.auth.default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
        api=build('drive','v3',credentials=creds,cache_discovery=False)
        req=api.files().get_media(fileId=x['id'],supportsAllDrives=True)
        with open(dest,'wb') as fh:
            dl=MediaIoBaseDownload(fh,req,chunksize=16*1024*1024)
            done=False
            while not done:
                _,done=dl.next_chunk(num_retries=5)
        print('Downloaded',dest.name,flush=True)
    assert dest.stat().st_size==expected,(dest,expected,dest.stat().st_size)
    if x.get('md5Checksum'):
        assert file_md5(dest)==x['md5Checksum'],f'MD5 mismatch: {dest.name}'
    return dest


## fastp merge → 정확한 서열별 Count

중간 merged FASTQ는 Colab의 로컬 임시 디스크에 기록합니다. 이는 1개 샘플에서 FIFO deadlock·압축 파이프라인 오류를 피하는 안정성을 우선한 구성입니다. `fastp` 옵션은 이전 결과와 일치합니다.


In [ ]:
def inspect_csv(path):
    n=total=invalid=0
    with gzip.open(path,'rt') as fh:
        reader=csv.DictReader(fh)
        assert reader.fieldnames==['sequence','read_count','length_bp'],reader.fieldnames
        for row in reader:
            s=row['sequence']
            cnt=int(row['read_count'])
            length=int(row['length_bp'])
            if not s or length!=len(s) or cnt<=0:
                invalid+=1
            n+=1
            total+=cnt
    assert invalid==0, f'Invalid rows: {invalid}'
    return n,total

if RESULT.exists() and not OVERWRITE:
    unique,merged=inspect_csv(RESULT)
    print(f'[SKIP] Existing verified output: merged={merged:,}, unique={unique:,}')
else:
    t0=time.monotonic()
    with ThreadPoolExecutor(max_workers=2) as pool:
        fq1,fq2=list(pool.map(download,[r1[0],r2[0]]))
    t_download=time.monotonic()
    merged_fq=TMP/'merged.fq.gz'
    unmerged_r1=TMP/'unmerged_R1.fq.gz'
    unmerged_r2=TMP/'unmerged_R2.fq.gz'
    fastp_json=TMP/'fastp.json'
    cmd=[
        'fastp','-i',str(fq1),'-I',str(fq2),
        '--merge','--merged_out',str(merged_fq),
        '-o',str(unmerged_r1),'-O',str(unmerged_r2),
        '--disable_adapter_trimming','--disable_quality_filtering',
        '--disable_length_filtering','--disable_trim_poly_g',
        '--overlap_len_require','20',
        '--overlap_diff_limit','5',
        '--overlap_diff_percent_limit','20',
        '--thread',str(FASTP_THREADS),
        '--json',str(fastp_json),'--html',str(TMP/'fastp.html')
    ]
    print('Running fastp merge...',flush=True)
    with open(TMP/'fastp.stderr.log','w') as log:
        subprocess.run(cmd,check=True,stderr=log)
    t_merge=time.monotonic()
    print(f'fastp complete, elapsed: {t_merge-t_download:.1f}s',flush=True)
    partial=TMP/(RESULT.name+'.partial')
    sorter=r'''set -o pipefail
mkdir -p "$3"
gzip -dc "$1" | awk 'NR%4==2 {print}' | LC_ALL=C sort --parallel=4 -S "$4" -T "$3" | LC_ALL=C uniq -c | awk 'BEGIN{print "sequence,read_count,length_bp"} {n=$1;$1="";sub(/^ /,"");print $0 "," n "," length($0)}' | gzip -1 > "$2"
'''
    subprocess.run(['bash','-n','-c',sorter],check=True)
    print('Sorting and exact-sequence counting...',flush=True)
    subprocess.run(
        ['bash','-c',sorter,'bash',str(merged_fq),str(partial),str(TMP/'sort_temp'),SORT_RAM],
        check=True
    )
    unique,merged=inspect_csv(partial)
    t_count=time.monotonic()
    report=json.loads(fastp_json.read_text())
    raw_pairs=int(report['summary']['before_filtering']['total_reads'])//2
    reported=report.get('merged_and_filtered',{}).get('total_reads')
    if reported is not None:
        assert merged==int(reported), f'Count mismatch: {merged} vs {reported}'
    assert 0<=merged<=raw_pairs
    print(f'Validation PASS: merged={merged:,}, unique={unique:,}',flush=True)
    staged=OUT/(RESULT.name+'.partial')
    shutil.copyfile(partial,staged)
    assert inspect_csv(staged)==(unique,merged)
    staged.replace(RESULT)
    import pandas as pd
    pd.DataFrame([{
        'sample':SAMPLE,'total_read_pairs':raw_pairs,
        'merged_reads':merged,'unique_sequences':unique,
        'merge_rate_percent':round(100*merged/max(raw_pairs,1),4),
        'download_seconds':round(t_download-t0,1),
        'merge_seconds':round(t_merge-t_download,1),
        'unique_count_seconds':round(t_count-t_merge,1)
    }]).to_csv(SUMMARY,index=False)
    print('Saved:',RESULT,flush=True)
    # Keep inputs only if an error occurred; delete after fully validated result.
    shutil.rmtree(TMP,ignore_errors=True)


## 재검증

원본 BAM인 `PLA_6_13.hifi_reads.bam`은 이 노트북에 포함하지 않습니다. PacBio 전용 노트북을 사용하세요. 이 노트북은 **PCR_6_13 Illumina paired-end 1개만 처리**합니다.
